# Explore the Mirrored Database

The Azure SQL database is being **mirrored** into Fabric: its tables are
replicated into OneLake as Delta tables, automatically and continuously, with
no pipelines, no schedules, and no code.

This notebook reads those replicated tables straight from OneLake and runs this
sector's headline analytics on them. The table list and the query come from the
sector's `mirroring.json` spec.

In [ ]:
import base64, json

SPEC = json.loads(base64.b64decode("{{MIRRORING_SPEC_B64}}").decode("utf-8"))
WORKSPACE_ID = "{{WORKSPACE_ID}}"
MIRRORED_DB_ID = "{{MIRRORED_DB_ID}}"

def mirrored(table):
    path = (
        f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/"
        f"{MIRRORED_DB_ID}/Tables/dbo/{table}"
    )
    return spark.read.format("delta").load(path)

for t in SPEC["tables"]:
    name = t["name"]
    print(f"dbo.{name}: {mirrored(name).count():,} rows replicated")

In [ ]:
# Register each replicated table as a temp view (m_<table>) and run the
# sector's headline query straight against the mirrored data. Zero ETL happened.
for t in SPEC["tables"]:
    name = t["name"]
    mirrored(name).createOrReplaceTempView(f"m_{name}")

print(SPEC["explore"]["title"])
result = spark.sql(SPEC["explore"]["sql"])
result.show(20, truncate=False)

In [ ]:
# Show a sample of the table that 02_live_change will modify
lc = SPEC.get("liveChange", {})
if lc.get("table"):
    print(f"Sample of mirrored {lc['table']} (02_live_change modifies this table):")
    mirrored(lc["table"]).show(5, truncate=False)
print("\nEverything above is served from OneLake Delta files that Fabric keeps in")
print("sync with Azure SQL automatically. Open 02_live_change to watch a source-side")
print("change replicate in near real time.")